# Smart scheduling: validation and evaluation

Run each cell in order with Shift+Enter. Select the Python kernel from `experiments/pasan/.venv`. The experiments below use the application scheduler and synthetic scenarios. They are not real-world savings or evidence of appliance execution. Read [README.md](README.md) for the full protocol.

## 1. Setup
Start Jupyter inside the project repository. This cell locates the repository and loads the evaluation helpers.

In [ ]:
from pathlib import Path
import sys, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Application/Backend/app/features/smartscheduling/milp/optimizer.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter inside J26-DS-323 or set PROJECT_ROOT manually.')
WORK = PROJECT_ROOT / 'experiments/pasan/validation_evaluation'
sys.path.insert(0, str(WORK))
from evaluate import frame, audit, validation, evaluation, ApplianceJob, solve_solar_only
print('Project:', PROJECT_ROOT)
print('Python:', sys.executable)

## 2. Validate the optimizer
Five small cases compare the solver with an independent exhaustive search and reconstructed load. A pass supports these examples, not every possible input.

In [ ]:
validation_results = validation()
display(validation_results[['case', 'status', 'oracle', 'pass']])
print('Passed:', validation_results['pass'].sum(), '/', len(validation_results))
assert validation_results['pass'].all(), 'A case failed: inspect the audit details below.' 

In [ ]:
display(validation_results[['case', 'audit']])

## 3. Run existing integration and dynamic replay tests
These use synthetic or mocked input and are not field trials.

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([str(PROJECT_ROOT / 'Application/Backend'), str(PROJECT_ROOT / 'experiments/pasan'), env.get('PYTHONPATH', '')])
command = [sys.executable, '-m', 'unittest', 'app.features.smartscheduling.milp.test_historical_mode', 'app.features.smartscheduling.dynamic.test_replay', 'dynamic_rescheduling.test_measurements', '-v']
test_run = subprocess.run(command, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
print(test_run.stdout + test_run.stderr)
if test_run.returncode:
    raise RuntimeError('Tests failed; check the output and kernel dependencies.')

## 4. Create a synthetic example
Change the jobs, solar scale and safety factor, then rerun this section. The drop in realized solar is invented for this demonstration.

In [ ]:
jobs = [ApplianceJob('washer', .7, 60, '08:00', '15:00', '09:00', 2),
        ApplianceJob('pump', .8, 60, '09:00', '16:00', '10:00', 1),
        ApplianceJob('cooker', .9, 30, '10:00', '15:00', '12:00', 3)]
solar_scale, safety_factor = 1.5, .7
slots = np.arange(52)
forecast = solar_scale * np.maximum(0, np.sin(np.pi * (slots - 4) / 44))
realized = forecast.copy()
realized[16:22] *= .35  # artificial shortfall from 10:00 to 11:30
solar = frame(forecast)
display(pd.DataFrame([vars(j) for j in jobs]))

In [ ]:
times = pd.to_datetime(solar.timestamp, utc=True).dt.tz_convert('Asia/Colombo')
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, forecast, label='Synthetic forecast')
ax.plot(times, forecast * safety_factor, '--', label='Usable forecast')
ax.plot(times, realized, label='Synthetic realized solar')
ax.set(title='Example solar profile', ylabel='Power (kW)', xlabel='Sri Lanka time')
ax.legend(); ax.grid(alpha=.25); fig.autofmt_xdate(); plt.show()

## 5. Solve and inspect the example
All jobs are mandatory. If the result is infeasible or not proven optimal, report the status; no schedule will be graphed.

In [ ]:
result = solve_solar_only(solar, jobs, safety_factor=safety_factor, time_limit_seconds=8)
print('Solver status:', result['status'])
print('Details:', result.get('reason', result.get('termination', '')))
if result['status'] == 'optimal':
    display(pd.DataFrame(result['schedule']))
    display(pd.DataFrame([audit(solar, jobs, result['starts'], safety_factor, actual=realized)]).T.rename(columns={0:'value'}))

In [ ]:
if result['status'] == 'optimal':
    load = result['trace']['scheduled_total_kw'].to_numpy()
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.step(times, load, where='post', label='Scheduled load')
    ax.step(times, forecast * safety_factor, where='post', label='Usable forecast')
    ax.step(times, realized, where='post', label='Synthetic realized solar')
    ax.fill_between(times, realized, load, where=load > realized, step='post', color='red', alpha=.25, label='Hypothetical shortfall')
    ax.set(title='Scheduled load and solar', xlabel='Sri Lanka time', ylabel='Power (kW)')
    ax.legend(); ax.grid(alpha=.25); fig.autofmt_xdate(); plt.show()
else:
    print('No optimal schedule to plot.')

## 6. Evaluate scenarios and safety factors
The preferred-start baseline does not enforce the solar constraint. Shortfall is the positive difference between load and synthetic realized solar, integrated over 15-minute slots; it is not measured grid import. Shift is priority × absolute start-time deviation. Missing values mean no optimal schedule was returned.

In [ ]:
results = evaluation()
display(results[['scenario','safety_factor','method','status','valid','violations','weighted_shift_min','peak_kw','realized_deficit_kwh','realized_deficit_slots','solve_seconds']])

In [ ]:
for metric, label in [('realized_deficit_kwh', 'Synthetic shortfall (kWh)'), ('weighted_shift_min', 'Priority-weighted start shift (minutes)')]:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
    for ax, scenario in zip(axes, ['sunny', 'mixed', 'cloudy']):
        subset = results[results.scenario == scenario]
        for method, group in subset.groupby('method'):
            group = group.sort_values('safety_factor')
            ax.plot(group.safety_factor, group[metric], marker='o', label=method)
        ax.set(title=scenario.title(), xlabel='Safety factor', xticks=[.7,.9,1.0]); ax.grid(alpha=.25)
    axes[0].set_ylabel(label); axes[-1].legend(); fig.suptitle(label); fig.tight_layout(); plt.show()

## 7. Inspect available measured PV data
This is a data-quality overview, not a performance evaluation. The backend assumes `pac` is in watts; verify this and the sampling coverage. The file does not include archived issue-time forecasts or appliance execution.

In [ ]:
measurement_path = PROJECT_ROOT / 'Application/Backend/app/data/smart_scheduling/meepe.csv'
if measurement_path.exists():
    measurements = pd.read_csv(measurement_path)
    measurements['timestamp'] = pd.to_datetime(measurements.dataTimestamp, unit='ms', utc=True).dt.tz_convert('Asia/Colombo')
    print('Rows:', len(measurements), '| Duplicates:', measurements.timestamp.duplicated().sum(), '| Missing:', measurements[['dataTimestamp','pac']].isna().sum().to_dict())
    print('Range:', measurements.timestamp.min(), 'to', measurements.timestamp.max())
    display(measurements.groupby(measurements.timestamp.dt.date).size().rename('readings').to_frame())
    fig, ax = plt.subplots(figsize=(11,4)); ax.plot(measurements.timestamp, measurements.pac / 1000, linewidth=1)
    ax.set(title='Measured PV readings (assuming watts)', ylabel='PV power (kW)', xlabel='Sri Lanka time'); ax.grid(alpha=.25); fig.autofmt_xdate(); plt.show()
else:
    print('Measurement file not found:', measurement_path)

## 8. Save results
This overwrites the two `notebook_*.csv` files. Keep the inputs and code version with your report.

In [ ]:
output_dir = WORK / 'results'
output_dir.mkdir(exist_ok=True)
validation_results.to_csv(output_dir / 'notebook_validation.csv', index=False)
results.to_csv(output_dir / 'notebook_controlled_evaluation.csv', index=False)
print('Saved to:', output_dir)

## 9. Conduct real-world validation without smart switches

Use the detailed [real-world validation guide](real_world_validation.md). Participants manually start appliances and log actual times; collect timestamped inverter readings and, if available, whole-house meter readings. Compare scheduled days with preferred-time days using a prespecified alternating or randomized assignment. Keep the original forecast and plan, record missed jobs and gaps, and report self-reported adherence separately from measured solar generation. Inverter data alone cannot verify appliance consumption or grid savings. The existing dynamic feature is an offline replay, not automatic live control.
